## Hyperparameter Tuning with Mlflow

### Setting up Optuna in the Environment

In [ ]:
%pip install optuna==4.7.0 optuna-integration==4.7.0

### Setup the Azure ML Client

In [ ]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

### Connect to the Azure ML Workspace

In [ ]:
import azureml.core
from azureml.core import Workspace

# Load the workspace from the saved config file
ws = Workspace.from_config()
print('Ready to use Azure ML {} to work with {}'.format(azureml.core.VERSION, ws.name))

### Fetch the Diabetes Dataset from the Data Asset

In [ ]:
from azureml.core import Workspace, Dataset

dataset = Dataset.get_by_name(ws, name='diabetes dataset')
diabetes_df = dataset.to_pandas_dataframe()

In [ ]:
display(diabetes_df)

### Preparing the Testing and Training Data with Normalization and Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import pandas as pd

numericFeatures = [
            "Pregnancies",
            "Glucose",
            "BloodPressure",
            "SkinThickness",
            "Insulin",
            "BMI",
            "DiabetesPedigreeFunction",
            "Age"
]

# Extract the feature matrix (similar to VectorAssembler)
numericFeaturesVector = diabetes_df[numericFeatures]

# Apply MinMax Scaling
scaler = MinMaxScaler()

normalizedFeatures = scaler.fit_transform(numericFeaturesVector)

# Create a scaled dataframe
scaledData = pd.DataFrame(
    normalizedFeatures,
    columns=numericFeatures
)

# Add the label column
scaledData["Outcome"] = diabetes_df["Outcome"]
train, test = train_test_split(scaledData, test_size=0.3, random_state=42)

### Creating Mlflow Experiment Function and Logging Model Runs

In [ ]:
import mlflow
mlflow.set_experiment("mlflow-hyperparam-finetuning")

### Defining the Objective Function

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

import optuna
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


def objective(trial):

    # Hyperparameters from Optuna
    max_depth = trial.suggest_int("MaxDepth", 1, 10)
    max_bins = trial.suggest_categorical("MaxBins", [10, 20, 30])  
    # Note: sklearn tree does not have maxBins like Spark

    numFeatures = [
        "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
        "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"
    ]

    X_train = train[numFeatures]
    y_train = train["Outcome"]

    X_test = test[numFeatures]
    y_test = test["Outcome"]

    with mlflow.start_run(nested=True):

        # sklearn pipeline
        pipeline = Pipeline([
            ("classifier", DecisionTreeClassifier(max_depth=max_depth, random_state=42))
        ])

        model = pipeline.fit(X_train, y_train)

        # Predictions
        preds = model.predict(X_test)

        accuracy = accuracy_score(y_test, preds)

        # Log parameters
        mlflow.log_param("MaxDepth", max_depth)
        mlflow.log_param("MaxBins", max_bins)

        # Log metric
        mlflow.log_metric("accuracy", accuracy)
        
        # Create signature
        input_example = pd.DataFrame([{
            "Pregnancies": 8, "Glucose": 85, "BloodPressure": 65, "SkinThickness": 29,
            "Insulin": 0, "BMI": 26.6, "DiabetesPedigreeFunction": 0.672, "Age": 34
        }])
        
        output_example = pd.DataFrame({"prediction": [1]})
        signature = infer_signature(input_example, output_example)
        
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            signature=signature,
            input_example=input_example
        )


    return accuracy

### Using Optuna for Hyperparameter Fine-Tuning

In [ ]:
import optuna
from optuna.integration.mlflow import MLflowCallback

# Create an Optuna study to maximize accuracy
with mlflow.start_run(run_name='optuna_hyperparameter_tuning'):
    sampler = optuna.samplers.TPESampler(seed=42)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name="diabetes_classification"
    )
    
    # Run optimization
    study.optimize(objective, n_trials=3)
    
    # Get best parameters
    print("Best param values: ", study.best_params)
    print("Best accuracy: ", study.best_value)
    
    # Log best parameters to parent run
    mlflow.log_params(study.best_params)
    mlflow.log_metric('best_accuracy', study.best_value)
    
    # Train BEST model
    X_train = train[numericFeatures]
    y_train = train["Outcome"]

    X_test = test[numericFeatures]
    y_test = test["Outcome"]

    best_model = DecisionTreeClassifier(
        max_depth=study.best_params["MaxDepth"],
        random_state=42
    )

    best_model.fit(X_train, y_train)

    preds = best_model.predict(X_test)

    # Create signature
    input_example = pd.DataFrame([{
            "Pregnancies": 8, "Glucose": 85, "BloodPressure": 65, "SkinThickness": 29,
            "Insulin": 0, "BMI": 26.6, "DiabetesPedigreeFunction": 0.672, "Age": 34
    }])
        
    output_example = pd.DataFrame({"prediction": [1]})
    signature = infer_signature(input_example, output_example)
        
    mlflow.sklearn.log_model(
            sk_model=best_model,
            artifact_path="model",
            signature=signature,
            input_example=input_example
    )

    # Display all trials
    print("\nAll trials:")
    for trial in study.trials:
        print(f"\nTrial {trial.number}: params={trial.params}, accuracy={trial.value}")

    